In [ ]:
!pip install tqdm

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import time

from google.colab import files
import os

from tensorflow.keras import layers
from tqdm.notebook import tqdm

In [ ]:
print("Faça o upload do arquivo kaggle.json com a API Key.")
files.upload()

In [ ]:
# Cria um diretório oculto chamado .kaggle
!mkdir -p ~/.kaggle

# Move o arquivo kaggle.json para o novo diretório
!mv kaggle.json ~/.kaggle/

# Altera as permissões do arquivo para evitar um aviso de segurança
!chmod 600 ~/.kaggle/kaggle.json

print("Baixando o dataset CelebA do Kaggle...")
!kaggle datasets download -d jessicali9530/celeba-dataset -q

print("Download concluído!")

In [ ]:
print("Descompactando imagens...")
!unzip -q celeba-dataset.zip

print("Dataset pronto para uso na pasta 'img_align_celeba/'!")

In [ ]:
# Define o caminho para a pasta com as imagens
PATH = 'img_align_celeba/img_align_celeba/*.jpg'

# Cria o dataset a partir dos arquivos locais
# list_files embaralha os caminhos dos arquivos por padrão
dataset = tf.data.Dataset.list_files(PATH)

# Pega um único caminho de arquivo do nosso dataset
for file_path in dataset.take(1):
  # Carrega e decodifica a imagem
  image = tf.io.read_file(file_path)
  image = tf.image.decode_jpeg(image, channels=3)

  # Mostra a imagem
  plt.figure(figsize=(6, 6))
  plt.imshow(image)
  plt.title(f"Dimensões: {image.shape}")
  plt.axis('off')
  plt.show()

In [ ]:
# Parâmetros do Pipeline e Modelo
IMAGE_SIZE = 64
BATCH_SIZE = 64
BUFFER_SIZE = 10000 # Tamanho do buffer para embaralhar os dados
PATH = 'img_align_celeba/img_align_celeba/*.jpg'

In [ ]:
def preprocess_image(file_path):
    # Ler o arquivo
    image = tf.io.read_file(file_path)
    # Decodificar para um tensor de 3 canais (RGB)
    image = tf.image.decode_jpeg(image, channels=3)

    # Cortar a imagem para ficar quadrada (178x178)
    image = tf.image.crop_to_bounding_box(image, 20, 0, 178, 178)

    # 4. Redimensionar para o tamanho desejado (64x64)
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])

    # 5. Normalizar os pixels para o intervalo [-1, 1]
    image = (tf.cast(image, tf.float32) - 127.5) / 127.5

    return image

In [ ]:
# Criar um Dataset com os caminhos dos arquivos
list_ds = tf.data.Dataset.list_files(PATH, shuffle=False)

# Mapear a função de pré-processamento para cada arquivo.
images_ds = list_ds.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

# Configuração do pipeline
train_dataset = (
    images_ds
    .shuffle(BUFFER_SIZE) # Embaralha as imagens processadas
    .batch(BATCH_SIZE)    # Agrupa as imagens em lotes de 128
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

print("Pipeline de dados criado com sucesso!")
print(train_dataset)

In [ ]:
# Pega um único lote de imagens do  pipeline
for image_batch in train_dataset.take(1):

    plt.figure(figsize=(8, 8))
    plt.suptitle("Amostra de Imagens Pré-processadas")

    # Mostra as primeiras 16 imagens do lote
    for i in range(16):
        ax = plt.subplot(4, 4, i + 1)

        # Desnormaliza a imagem de [-1, 1] para [0, 1] para exibição
        img_to_show = (image_batch[i].numpy() + 1) / 2.0

        plt.imshow(img_to_show)
        plt.axis("off")

    plt.show()
    break

In [ ]:
# @title Criando o Gerador

def make_generator_model():
    model = tf.keras.Sequential()

    # Camada de entrada: Pega o vetor de ruído (100 dimensões) e o transforma
    # em um tensor base para começar o "upsampling". 4*4*1024 = 16384 neurônios.
    model.add(layers.Dense(4*4*1024, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Remodela o tensor para um formato de "imagem" inicial: 4x4x1024
    model.add(layers.Reshape((4, 4, 1024)))
    assert model.output_shape == (None, 4, 4, 1024) # A anotação None é para o batch size

    # Bloco de Upsampling 1: 4x4 -> 8x8
    # Conv2DTranspose aumenta a resolução.
    model.add(layers.Conv2DTranspose(512, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 8, 8, 512)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Bloco de Upsampling 2: 8x8 -> 16x16
    model.add(layers.Conv2DTranspose(256, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 16, 16, 256)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Bloco de Upsampling 3: 16x16 -> 32x32
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 32, 32, 128)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Bloco Final de Saída: 32x32 -> 64x64
    # A saída deve ter 3 canais (RGB).
    # A função de ativação 'tanh' garante que os pixels fiquem no intervalo [-1, 1],
    # o mesmo intervalo que foi usado para normalizar nossas imagens reais.
    model.add(layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 64, 64, 3)

    return model

# Cria uma instância do gerador
generator = make_generator_model()

generator.summary()

In [ ]:
# Cria um vetor de ruído aleatório (1 amostra, 100 dimensões)
noise = tf.random.normal([1, 100])

# Cria uma imagem a partir do ruído
generated_image = generator(noise, training=False)

# Mostra a imagem gerada
plt.imshow(generated_image[0, :, :, 0], cmap='gray')
plt.title("Saída do Gerador Não Treinado")
plt.axis('off')
plt.show()

In [ ]:
# @title Criando o Discrimidador

def make_discriminator_model():
    model = tf.keras.Sequential()

    # Camada de Entrada: Imagem 64x64x3
    # Bloco 1: Convolução 64x64 -> 32x32
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same',
                                     input_shape=[64, 64, 3]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3)) # Dropout ajuda a regularizar e evitar que o discriminador fique forte demais

    # Bloco 2: Convolução 32x32 -> 16x16
    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # Bloco 3: Convolução 16x16 -> 8x8
    model.add(layers.Conv2D(256, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    # Bloco Final de Classificação
    # Achatamos o tensor 3D para um vetor 1D
    model.add(layers.Flatten())
    # Camada de saída com um único neurônio. A saída é um logit bruto
    # pois a função de perda que usaremos é otimizada para logits.
    model.add(layers.Dense(1))

    return model

# Cria uma instância do discriminador
discriminator = make_discriminator_model()

discriminator.summary()

In [ ]:
# Classifica a imagem gerada
decision = discriminator(generated_image, training=False)

# Imprime a decisão
print("Decisão do Discriminador não treinado sobre a imagem falsa:")
print(decision)

In [ ]:
# @title Definindo as Funções de Perda

# A função de perda que usaremos compara a previsão do discriminador (logits)
# com os rótulos verdadeiros (1 para real, 0 para falso).
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# Perda do Discriminador
# O quão bem o discriminador consegue separar imagens reais de falsas?
def discriminator_loss(real_output, fake_output):
    # A perda para imagens reais: compara a previsão para imagens reais (real_output) com um array de 1s.
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)

    # A perda para imagens falsas: compara a previsão para imagens falsas (fake_output) com um array de 0s.
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)

    # A perda total é a soma das duas.
    total_loss = real_loss + fake_loss
    return total_loss


# Perda do Gerador
# O quão bem o gerador consegue enganar o discriminador?
def generator_loss(fake_output):
    # O objetivo do gerador é que o discriminador classifique suas imagens falsas como REAIS.
    # Por isso, comparamos a previsão do discriminador (fake_output) com um array de 1s.
    return cross_entropy(tf.ones_like(fake_output), fake_output)


# Usamos o otimizador Adam.
generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)

print("Funções de perda e otimizadores definidos com sucesso!")

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")

checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

print("Checkpoint configurado.")

In [ ]:
# @title Definição de um passo de treinamento

# Define as dimensões do vetor de ruído que o Gerador usará
noise_dim = 100

# A anotação compila a função, acelerando drasticamente o treinamento
@tf.function
def train_step(images):
    # Gera um lote de vetores de ruído aleatório
    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # O Gerador cria imagens falsas a partir do ruído
        generated_images = generator(noise, training=True)

        # O Discriminador avalia tanto as imagens reais quanto as falsas
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)

        # Calcula a perda para cada modelo usando as funções que definimos
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    # Calcula os gradientes para cada modelo
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    # Aplica os ajustes aos modelos
    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

    return gen_loss, disc_loss

In [ ]:
# Função de gerar imagens
def generate_and_save_images(model, epoch, test_input):
    predictions = model(test_input, training=False)
    fig = plt.figure(figsize=(6, 6))
    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        plt.imshow((predictions[i, :, :, :] + 1) / 2.0)
        plt.axis('off')
    plt.suptitle(f'Imagens geradas na Época {epoch}')
    plt.savefig(f'image_at_epoch_{epoch:04d}.png')
    plt.show()


# Loop de treinamento
def train(dataset, epochs):
    num_examples_to_generate = 16
    seed = tf.random.normal([num_examples_to_generate, noise_dim])

    print("="*60)
    print(f"🚀 INICIANDO O TREINAMENTO DA GAN 🚀")
    print(f"Total de Épocas: {epochs}")
    print(f"Tamanho do Lote: {BATCH_SIZE}")
    print("="*60)

    for epoch in range(epochs):
        start = time.time()

        # Inicializa as perdas da época para calcular a média no final
        epoch_gen_loss_avg = tf.keras.metrics.Mean()
        epoch_disc_loss_avg = tf.keras.metrics.Mean()

        # Cria a barra de progresso com tqdm
        pbar = tqdm(dataset, desc=f"Época {epoch + 1}/{epochs}", unit=" lote")

        for image_batch in pbar:
            # Executa o passo de treino e obtém as perdas
            gen_loss, disc_loss = train_step(image_batch)

            # Atualiza a barra de progresso com as perdas do lote atual
            pbar.set_postfix({"Loss Gerador": f"{gen_loss:.4f}", "Loss Disc.": f"{disc_loss:.4f}"})

            # Acumula as perdas para calcular a média da época
            epoch_gen_loss_avg.update_state(gen_loss)
            epoch_disc_loss_avg.update_state(disc_loss)

        print(
            f"Época {epoch + 1} | "
            f"Loss Gerador (média): {epoch_gen_loss_avg.result():.4f} | "
            f"Loss Disc. (média): {epoch_disc_loss_avg.result():.4f} | "
            f"Tempo: {time.time()-start:.2f}s"
        )

        print("Gerando imagens de amostra ao final da época...")
        generate_and_save_images(generator, epoch + 1, seed)

        if (epoch + 1) % 10 == 0:
            print(f"Salvando checkpoint para a época {epoch + 1}...")
            checkpoint.save(file_prefix=checkpoint_prefix)

        print("="*60)

    print("\n🎉 Treinamento concluído! 🎉")

In [ ]:
# @title Treinamento

EPOCHS = 10

train(train_dataset, EPOCHS)

In [ ]:
# @title Gerar Imagens

def generate_from_checkpoint(generator_model, num_images=16, noise_dim=100):
    print("Iniciando o processo de geração de imagens...")

    checkpoint_dir = './training_checkpoints'

    # Cria um objeto Checkpoint que mapeia as variáveis do modelo
    checkpoint = tf.train.Checkpoint(generator=generator_model)

    # Encontra o arquivo do checkpoint mais recente no diretório
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)

    if latest_checkpoint:
        # Restaura os pesos salvos para o modelo
        status = checkpoint.restore(latest_checkpoint)
        status.expect_partial()
        print(f"Checkpoint restaurado com sucesso de: {latest_checkpoint}")
    else:
        print("❌ ERRO: Nenhum checkpoint encontrado no diretório especificado.")
        print("Certifique-se de que o treinamento salvou os arquivos ou que o caminho está correto.")
        return

    print(f"\nGerando {num_images} novas imagens...")

    # Cria um lote de vetores de ruído aleatório como entrada para o gerador
    noise = tf.random.normal([num_images, noise_dim])

    # Gera as imagens usando o modelo carregado
    predictions = generator_model(noise, training=False)

    # Calcula o tamanho da grade para exibir as imagens (ex: 4x4 para 16 imagens)
    grid_size = int(np.ceil(np.sqrt(num_images)))

    fig = plt.figure(figsize=(grid_size * 2, grid_size * 2))
    plt.suptitle("Imagens Geradas pelo Modelo Final", fontsize=16)

    for i in range(predictions.shape[0]):
        plt.subplot(grid_size, grid_size, i+1)
        # Desnormaliza a imagem de [-1, 1] para [0, 1] para poder exibi-la
        plt.imshow((predictions[i, :, :, :] + 1) / 2.0)
        plt.axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Defina quantas imagens criar
NUM_IMAGENS_PARA_GERAR = 16

generate_from_checkpoint(generator, num_images=NUM_IMAGENS_PARA_GERAR)